In [ ]:
import glob
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm

In [ ]:
# Get gene trait associations
RAP_DIR = 'project-REDACTED:/processed_data/REGENIE_results'

# ASSOC_FILE = 'loftee_mac20_associations_bh_corrected.parquet'
# ASSOC_FILE = 'regenie_127phenotypes_lofteeHC_mac20_EUR.parquet'
ASSOC_FILE = 'regenie_127phenotypes_lofteeHC_mac20_EUR_miss20per.parquet'

LOCAL_DIR = 'PATH_TO_FILE'

!dx download {RAP_DIR}/{ASSOC_FILE} -o {LOCAL_DIR}

gene_trait_df = (
    pl.read_parquet(f'{LOCAL_DIR}/{ASSOC_FILE}')
    .filter(pl.col('pval_fdr')<=0.05)
    .select(['region', 'phenotype', 'pval_fdr'])
)

# CORR_FILE = 'regenie_127phenotypes_mac20_lofteeHC_EUR_correlations.parquet'
# !dx download {RAP_DIR}/{CORR_FILE} -o {LOCAL_DIR}

# loftee_corrs = (
#     pl.read_parquet(f'{LOCAL_DIR}/{CORR_FILE}')
#     .with_columns(
#         loftee_corr = pl.col('correlation'),
#         loftee_corr_abs = pl.col('correlation').abs(),
#         loftee_corr_dir = pl.col('correlation')/pl.col('correlation').abs(),
#     )
#     .select(['region', 'phenotype', 'loftee_corr', 'loftee_corr_abs', 'loftee_corr_dir']) 
# )

# gene_trait_df = (
#     gene_trait_df
#     .join(loftee_corrs, on=['region', 'phenotype'], how='inner')
#     .drop_nans()
#     .sort('loftee_corr_abs', descending=True)
#     .unique(subset=["region"], keep="first", maintain_order=True)
# )
gene_trait_df

In [ ]:
# EUR unrelated individuals

!dx download project-REDACTED:/processed_data/sample_lists/unrelated_cauc_samples_3rd_degree.csv -o PATH_TO_FILE

unrel_eur_samples = pl.read_csv('PATH_TO_FILE')['eid'].cast(pl.Utf8).to_list()
unrel_eur_samples[:5]

In [ ]:
from scipy.special import ndtri

c = 3/8  # Blom's constant for inverse normal transformation (prevents infinite values at the tails)

# Download phenotypes: covariates and PRS corrected
!dx download project-REDACTED:/processed_data/phenotypes/corrected_cov_PRS_traits_EUR.parquet -o PATH_TO_FILE

phenos = (
    pl.read_parquet('PATH_TO_FILE')
    .rename({'individual':'sample'})
    .filter(pl.col('sample').is_in(unrel_eur_samples))
)

# phenos
long_phenos_int = (
    phenos
    .unpivot(
        index='sample',
        on=gene_trait_df['phenotype'].unique().to_list(),
        variable_name='phenotype',
        value_name='pheno_value'
    )
    .drop_nulls()

    .lazy()  # Use Lazy mode for better memory/query optimization
    .with_columns(
        # Calculate rank and group size using native Rust engine
        r = pl.col("pheno_value").rank().over("phenotype"),
        n = pl.len().over("phenotype")
    )
    .with_columns(
        # Calculate the INT value calling ndtri ONCE on the whole column
        pheno_value_int = ((pl.col("r") - c) / (pl.col("n") - 2*c + 1)).map_batches(ndtri)
    )
    .drop(["r", "n"]) # Clean up temporary columns
    .collect()
)

print(long_phenos_int['phenotype'].value_counts(sort=True))
long_phenos_int

In [ ]:
long_phenos_int.filter(pl.col('phenotype')=='standing_height_int').select(pl.col('pheno_value').std())

In [ ]:
long_phenos_int.filter(pl.col('phenotype')=='standing_height_int').select(pl.col('pheno_value_int').std())

In [ ]:
mac = 20

RAP_ANNO_DIR = "project-REDACTED:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated"

# ANNO_FILE = "annotations_fillna_ukbgym.parquet"
ANNO_FILE = "annotations_fillna_ukbgym_with_mane.parquet"

!dx download {RAP_ANNO_DIR}/{ANNO_FILE} -o {LOCAL_DIR}/{ANNO_FILE}

id_list = (
    pl.scan_parquet(f'{LOCAL_DIR}/{ANNO_FILE}')
    .filter(
        pl.col('region').is_in(gene_trait_df.select('region').unique().to_series()),
        pl.col('mac_ukb')<=mac,
    )
    .select('id')
    .unique()
    .collect()
)

id_list

In [ ]:
# Download genotype (long gt) file
!dx download project-REDACTED:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated/gt_long.parquet -o PATH_TO_FILE

long_gt = (
    pl.scan_parquet('PATH_TO_FILE')
    .select(['id', 'sample', 'gt'])
    .filter(
        pl.col('gt')==1,
        pl.col('sample').is_in(unrel_eur_samples),
    )
    .join(
        id_list.lazy(),
        on='id',
        how='semi'
    )

    .collect()
)

long_gt

In [ ]:
import math

output_dir = 'PATH_TO_FILE'
!mkdir -p {output_dir}

pheno_list = long_phenos_int['phenotype'].unique().to_list()
CHUNK_SIZE = 10
num_phenos = len(pheno_list)
num_chunks = math.ceil(num_phenos / CHUNK_SIZE)

# Process in Batches
for i in tqdm(range(0, num_phenos, CHUNK_SIZE)):
    # 1. Define the current batch of genes
    chunk_phenos = pheno_list[i : i + CHUNK_SIZE]

    print(f"Processing chunk starting at index: {i}")
    (
        long_phenos_int.lazy()
        .filter(pl.col('phenotype').is_in(chunk_phenos))
        .join(
            long_gt.lazy(),
            on='sample',
            how='inner'
        )
        .group_by(['id', 'phenotype'])
        .agg(
            n_individuals = pl.len().cast(pl.Int32),
            mean_pheno_value = pl.col('pheno_value_int').mean().cast(pl.Float32),
            std_pheno_value = pl.col('pheno_value_int').std().cast(pl.Float32),
        )
        
        # .with_columns(
        #     # Calculate rank and group size using native Rust engine
        #     r = pl.col("mean_pheno_value").rank().over("phenotype"),
        #     n = pl.len().over("phenotype")
        # )
        # .with_columns(
        #     # Calculate the INT value calling ndtri ONCE on the whole column
        #     mean_pheno_value_int = ((pl.col("r") - c) / (pl.col("n") - 2*c + 1)).map_batches(ndtri).cast(pl.Float32)
        # )
        # .drop(["r", "n"]) # Clean up temporary columns

        # .with_columns(
        #     mean_pheno_value_rank=pl.col('mean_pheno_value')
        #         .rank(method="max")
        #         .over('phenotype')
        #         .cast(pl.Float32),
        # )
        # .with_columns(
        #     mean_pheno_value_ptile=(
        #         pl.col('mean_pheno_value_rank') / pl.len().over('phenotype')
        #     ).cast(pl.Float32),
        # )

        .sink_parquet(f'{output_dir}/tmp_appv_chunk_{i}.parquet')
    )


In [ ]:
tmp = pl.read_parquet(f'{output_dir}/tmp_appv_chunk_0.parquet')
tmp

In [ ]:
(
    tmp
    .filter(pl.col('phenotype')=='apolipoprotein_b_int')
    .select(pl.col('mean_pheno_value').std())
)

In [ ]:
output_dir = 'PATH_TO_FILE'
small_output_file = "PATH_TO_FILE"

# 1. Get list of files manually
files = glob.glob(f'{output_dir}/*.parquet')
print(f"Found {len(files)} files.")

# 2. Create a list of LazyFrames
lfs = [pl.scan_parquet(f) for f in files]

# 3. Concatenate with relaxation
appv_big = pl.concat(lfs, how="vertical_relaxed")


id_region = pl.scan_parquet(f'{LOCAL_DIR}/{ANNO_FILE}').select(['id', 'region']).unique()

(
    appv_big
    .join(id_region, on='id', how='inner')
    .join(gene_trait_df.lazy(), on=['region', 'phenotype'], how='inner')
    .drop(['region', 'pval_fdr'])
    .unique(subset=['id', 'phenotype'])          # ← deduplicate
    .sink_parquet(small_output_file, engine='streaming')
)

In [ ]:
import polars as pl
small_output_file = "PATH_TO_FILE"
tmp = pl.read_parquet(small_output_file)
# tmp.head().collect()
tmp

In [ ]:
# !dx upload {combined_output_file} --path project-REDACTED:/processed_data/ukbgym/avg_pheno_per_var/

!dx upload {small_output_file} --path project-REDACTED:/processed_data/ukbgym/avg_pheno_per_var/